# Prediction of overall survival patients with Myeloid Leukemia

## Importing Modules

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Importing Data

### Clinical dataset

In [ ]:
X_clinical = pd.read_csv('../data/X_train/clinical_train.csv')
X_clinical = X_clinical.set_index('ID')
X_clinical

In [ ]:
print(f'Shape of the clinical dataset: {X_clinical.shape}')
X_clinical.dtypes

### Molecular dataset

In [ ]:
X_molecular = pd.read_csv('../data/X_train/molecular_train.csv')
X_molecular = X_molecular.set_index('ID')
X_molecular

In [ ]:
print(f'Shape of the molecular dataset: {X_molecular.shape}')
X_molecular.dtypes

### Labels dataset

In [ ]:
Y = pd.read_csv('../data/target_train.csv')
Y = Y.set_index('ID')
Y

In [ ]:
print(f'Shape of the labels dataset: {Y.shape}')
Y.dtypes

### ID coherence

In [ ]:
unique_id = X_clinical.index.is_unique, X_molecular.index.is_unique, Y.index.is_unique
print(f'The ID is unique: X_clinic=  {unique_id[0]}, X_molecular= {unique_id[1]}, Y= {unique_id[2]}')
print(f'Number of non matching ID between X_clinic and Y: {len(Y.index.difference(X_clinical.index))}')
print(f'Number of non matching ID between X_molecular and Y: {len(Y.index.difference(X_molecular.index))}')

ID coherence in the datasets:
- In the clinical and labels datasets, IDs are unique.
- In the molecular dataset, IDs are not unique, a patient can have several rows.
- All patients in the labels dataset have an instance in the clinical datasets.
- Some patients don't have an instance in the molecular datasets.\
\
**A patient can have none, one or multiple mutations.**



## Labels: OS_YEARS & OS_STATUS

### Missing data in the labels

Calculating the proportion of missing data.

In [ ]:
Y_missing = pd.DataFrame({'Number of missing data' : Y.isna().sum(), 'Proportion of missing data (%)' : (Y.isna().sum()/Y.shape[0])*100})
Y_missing

In [ ]:
Y_missing_matrix = pd.crosstab(Y['OS_YEARS'].isna(), Y['OS_STATUS'].isna())
Y_missing_matrix

- 150 data are missing in both OS_YEARS and OS_STATUS.
- Every missing data are missing from both labels.
- Missing data represents 4.5% of the dataset. \
\
**We are removing the missing data and we are storing the new labels in Yv.**

In [ ]:
Yv = Y.dropna()
Yv

### Values and coherence between OS_YEARS and OS_STATUS

1. OS_STATUS takes 0 or 1
2. OS_YEARS should be positive
3. Patients with no follow-up or who were included at the very end of the study: OS_YEARS = 0 and OS_STATUS = 0
4. Distribution of survivability duration per status

In [ ]:
# 1. OS_STATUS takes 0 or 1
print(Yv['OS_STATUS'].value_counts(dropna=False))

In [ ]:
# 2. OS_STATUS >= 0
print(Yv['OS_YEARS'].describe())
print((Yv['OS_YEARS'] < 0).sum())

In [ ]:
# 3. Patient not under follow-up
print(Yv[Yv['OS_YEARS'] == 0]['OS_STATUS'].value_counts())

In [ ]:
# 4.
print(Yv.groupby('OS_STATUS')['OS_YEARS'].describe())
sns.histplot(data=Yv, x='OS_YEARS', hue='OS_STATUS', bins=44)

### Censorship

The data are right-censored. \
OS_YEARS represents the duration of the follow-up. A patient can be included in the study within the all duration of the study ~22 years. OS_STATUS indicates if the patient has survived at the end of the follow-up. However, we do not know if the follow-up has ended because the study has ended or for another reason (eg. the follow-up is done somewhere else, ...).

1. We are studying the censorship rate.
2. We are studying the distribution of the censorship in the data.

In [ ]:
# 1. Censorship rate
censoring_rate = (Yv['OS_STATUS'] == 0).mean()
print(f'Censorship rate: {censoring_rate:.1%}')

In [ ]:
# 2. Distribution of the censorship
censoring_distribution = Yv
censoring_distribution['bin'] = pd.cut(censoring_distribution['OS_YEARS'], bins = [0, 0.5, 1, 1.5, 2, 2.5, 3, 5, 7, 10, 25])
summary = censoring_distribution.groupby('bin', observed=True)['OS_STATUS'].agg(n='count', censoring_rate=lambda s: (s == 0).mean().round(2)*100)
summary

In [ ]:
sns.histplot(Yv, x=Yv['OS_YEARS'], hue=Yv['OS_STATUS'], bins=44, multiple='stack')

We do not have the date of inclusion, therefore it is hard to tell if the censorship is due to the end of the study or if the patients stopped participating in the study.
- Censorship rate: 49,6%.
- Censorship distribution:
  - 49% before 6 month
  - then, a regular rising between 6 month and 7 years (67 % at 7 years)

The censorship distribution reveal a high proportion of censorship occurring within the first 6 months, which probably means that these were patients who, for one reason or another (short follow-up care, change of mind, ...), stopped participating in the study. \
The distribution for the rest of the group suggest that the censorship correspond to a patient alive at the end of the study.

## Clinical data

In [ ]:
print(X_clinical['CENTER'].value_counts())
print(f'Cytogenetics cardinality: {X_clinical['CYTOGENETICS'].nunique()}')       # Karyotype cardinality
print(X_clinical['CYTOGENETICS'].value_counts().head(10))

### Missing data

We are studying the missing data by feature.

In [ ]:
X_clinic_miss_by_feat = pd.DataFrame({'number of missing data': X_clinical.isna().sum(), 'Proportion of missing data (%)': (X_clinical.isna().sum() / X_clinical.shape[0]) * 100})
X_clinic_miss_by_feat


Some feature have a lot of missing data (eg. MONOCYTES). \
We are studying the missing data by CENTER to identify if these missing data are related to the CENTER where the patient was followed.

In [ ]:
X_clinic_miss_by_center = X_clinical.groupby('CENTER')[X_clinical.columns.drop('CENTER', 'ID')].apply(lambda g: g.isna().mean() * 100)
X_clinic_miss_by_center

In [ ]:
sns.heatmap(X_clinic_miss_by_center, annot=True)

- There is a lot of missing data for the MONOCYTES (18,1%) CYTOGENETICS (11,6%) and WBC (8,2%) features.
- Some CENTERS like REL and VU did not collect any MONOCYTES data
- Missing data are over-represented in some CENTER: HMS & DUTH
- Missing data in the MONOCYTES, WBC, ANC and PLT features seems to vary by CENTER
- Missing data in the BM_BLAST and HB features seems to be more randomly distributed \
\
**In the clinical dataset, missing data are not randomly distributed. We must take this into account when replacing missing data.**

### Distribution

#### Hematological parameters distribution

In [ ]:
X_clinical.describe()

In [ ]:
hemo_var = ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES', 'HB', 'PLT'] # hematological parameters

fig, axes = plt.subplots(2,3, figsize= (20, 10))
axes = axes.flatten()

# Distribution of hematological parameters
for var, ax in zip(hemo_var, axes):
    sns.histplot(data=X_clinical, x=var, bins=30, kde=True, ax=ax)
    ax.set_title(var)

Distribution:
- BM_BLAST, WBC, ANC, MONOCYTES and PLT are asymmetrical to the left.
- HB is symmetrical.
- MONOCYTES, ANC and WBC are very concentrated on the low values.
- PLT and BM_BLAST are less concentrated.


#### Hematological parameters distribution per OS_STATUS

In [ ]:
fig, axes = plt.subplots(2,3, figsize= (20, 10))
axes = axes.flatten()

# Distribution of hematological parameters per status
for var, ax in zip(hemo_var, axes):
    sns.histplot(data=X_clinical, x=var, hue=Yv['OS_STATUS'], bins=30, kde=True, ax=ax)
    ax.set_title(var)


- For BM_BLAST, the non-deceased patients are concentrated in the lowest category.
- For ANC, WBC and MONOCYTES the distribution between deceased and non-deceased sems to be the same.
- For PLT deceased patients are concentrated in the lower categories.
- For HB deceased patients sems to be more concentrated in the lower values and the non-deceased sems to be more concentrated in the higher values.


#### Outliners

Some patients have very high values for WBC, MONOCYTES, ANC and BM_BLAST but the median of those parameters are small. The data are concentrated in the lef side. \
We are studying the outliners to see if they are corresponding to a certain type of population or if they are corresponding to measurement mistakes. \
\
Lets see what proportion of the effective stands in the tail.
To do so we are using the 1,5*IQR criteria.

In [ ]:
queue_rate ={}
for var in hemo_var:
    q1, q3 = X_clinical[var].quantile([.25, .75])
    iqr = q3 - q1
    out = X_clinical[(X_clinical[var] > q3 + 1.5*iqr) | (X_clinical[var] < q1 - 1.5*iqr)] # outliners
    queue_rate[var] = (len(out) / len(X_clinical) * 100)
pd.DataFrame([queue_rate])

More than 3% of the effective are outliners for BM_BLAST, WBC, ANC, PLT and MONOCYTES. These values do not appear to be an error but likely describe subgroups of patient. Even if they are very high compared to the median.

Passing into log scale to see if it will reduce the number of outliners. \
We are drawing box plots with log scale.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()

for var, ax in zip(hemo_var, axes):
    sns.boxplot(data=X_clinical, x=var, ax=ax)
    ax.set_title(var)
    ax.set_xscale('log') # the data are asymmetrical, the log scale

The box plot shows that the queue sems to be very heavy for BM_BLAST, WBC, ANC, MONOCYTES and PLT, even in log scale.

Lets study the proportion of WBC values in the tail by CENTERS.

In [ ]:
q1, q3 = X_clinical['WBC'].quantile([.25, .75])
iqr = q3 - q1
print(q3 + 1.5*iqr)
X_clinical['WBC_out'] = (X_clinical['WBC'] > q3 + 1.5*iqr)
X_clinical.groupby('CENTER').agg(wbc_out_pct=('WBC_out', 'mean')).sort_values('wbc_out_pct', ascending=False)

Some centers are concentrating the higher values for WBC but none sems to really concentrated all the tail value, we can discard a record error or an error of measurement from a specific center.

Let's study the difference between the mean for all patients and the mean of patients with high value for WBC (the hyper group).

In [ ]:
hemo_vars = ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES', 'HB', 'PLT']
df_plot = X_clinic
df_plot['HYPER'] = df_plot['WBC'] > 50

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for var, ax in zip(hemo_vars, axes):
    sns.boxplot(data=df_plot, x='HYPER', y=var, ax=ax)
    ax.set_yscale('log')
    ax.set_title(var)

### Discrete features

#### CENTER

In [ ]:
print(f'Number of missing data in centers: {X_clinical['CENTER'].isna().sum()}')
print(f'Number of different centers: {X_clinical['CENTER'].nunique()}')
center_counts = X_clinical['CENTER'].value_counts()
center_counts

qWe have 23 different centers. The number of patients per center varies widely (between 6 and 900).

Crossing CENTER with OS_STATUS.

In [ ]:
center_x_os_status = pd.crosstab(X_clinical['CENTER'], Y['OS_STATUS'], normalize='index')
center_x_os_status

We are calculating
- the deceased rate and the censorship rate per center. Then, for each center, we are calculating
- the gap between the deceased rate for this center and the global deceased rate.
- the standard deviation (Sigma). \
Centers with a gap > 2*sigma are centers where a deceased rate variation compared to the global deceased rate can't be explain by the noise, it is a real difference.

In [ ]:
temp = X_clinical.join(Y)
print(temp.shape)
deceased_per_center = temp.groupby('CENTER')['OS_STATUS'].agg(n= 'count', Deceased= 'sum')
# Adding Deceased rate
deceased_per_center['Deceased rate'] = deceased_per_center['Deceased']/deceased_per_center['n']
# Adding Censorship rate
deceased_per_center['Censorship rate'] = 1 - deceased_per_center['Deceased rate']
# Adding the gap between the center deceased rate and the global deceased rate
p_global = temp['OS_STATUS'].mean()
deceased_per_center['Gap'] = (deceased_per_center['Deceased rate'] - p_global).abs()
# Adding standard deviation per center
deceased_per_center['Sigma'] = (p_global * (1 - p_global) / deceased_per_center['n']) ** 0.5
# Comparing the gap to the standard deviation
deceased_per_center['Real difference'] = deceased_per_center['Gap'] > 2 * deceased_per_center['Sigma']

deceased_per_center

The deceased rate per center is heterogeneous. \
For the center with a real difference, we have two categories :
 - the ones with high deceased rate and low censorship rate
 - the ones with low deceased rate and high censorship rate \
\
But we cannot determine if the high censorship rate is due to a short follow-up (the patient is still alive because the follow-up started at the end of the study) or if the follow-up was longer and the patient survived.

Let's calculate the median of the follow-up per center.

In [ ]:
follow_up = temp[temp['OS_STATUS'] == 0].groupby('CENTER')['OS_YEARS'].agg(follow_up_median='median', n_censordhip='count')

deceased_per_center['Follow_up_median'] = temp[temp['OS_STATUS'] == 0].groupby('CENTER')['OS_YEARS'].agg(follow_up_median='median')

deceased_per_center[['Follow_up_median', 'Censorship rate']]

In [ ]:
deceased_per_center['Follow_up_median'] = temp[temp['OS_STATUS'] == 0].groupby('CENTER')['OS_YEARS'].agg(follow_up_median='median')

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(deceased_per_center['Censorship rate'], deceased_per_center['Follow_up_median'],
           s=deceased_per_center['n'] * 3, alpha=0.5, edgecolor='black')

for center, row in deceased_per_center.iterrows():
    ax.annotate(center, (row['Censorship rate'], row['Follow_up_median']),
                fontsize=8, xytext=(4, 4), textcoords='offset points')

ax.set_xlabel('Censorship rate')
ax.set_ylabel('Follow-up median (years)')
ax.set_title('Censorship vs follow-up median per center (size = effectives)')

The more the censorship rate is high, the more the follow-up median is short. The more the censorship rate is high for a center, the less long the follow-up is. The high censorship rate in some center is due to a short follow-up in this center. A ceter with long follow-up see more deceased.

### CYTOGENETICS

In [ ]:
cyto = X_clinical['CYTOGENETICS']

print(f'Number of missing data in cytogenetics: {X_clinical['CYTOGENETICS'].isna().sum()}')
print(f'Number of unique karyotype: {X_clinical['CYTOGENETICS'].nunique()}')
cyto_counts = X_clinical['CYTOGENETICS'].value_counts()
cyto_counts

There is 387 missing data in CYTOGENETICS. There is 1194 unique Karyotype.